In [1]:
from Bio.Blast import NCBIWWW
from Bio import SeqIO
import io

In [ ]:
#result_handle = NCBIWWW.qblast("blastp", "nr", fasta_sequence)

In [2]:
#validate fasta from MODIDB

from Bio import SeqIO

def validate_fasta_file(filepath):
    """Validates a FASTA file using Biopython."""
    try:
        with open(filepath, "r") as handle:
            for record in SeqIO.parse(handle, "fasta"):
                # Basic check: record.id and record.seq should be available
                # More rigorous checks can be added here, e.g., checking sequence characters
                pass
        print(f"FASTA file '{filepath}' appears to be valid.")
        return True
    except Exception as e:
        print(f"Error validating FASTA file '{filepath}': {e}")
        return False

# Example usage:
validate_fasta_file("/home/wenyuantong/Desktop/database/mobidb_search_2025-10-07T22-18-23.fasta")

FASTA file '/home/wenyuantong/Desktop/database/mobidb_search_2025-10-07T22-18-23.fasta' appears to be valid.


True

In [12]:
from Bio import SeqIO

fasta_file = "/home/wenyuantong/Desktop/database/mobidb_search_2025-10-07T22-18-23.fasta"

#record = SeqIO.read(fasta_file, "fasta")

#with open("/home/wenyuantong/Desktop/database/mobidb_search_2025-10-07T22-18-23.fasta", "r") as handle:
    #for record in SeqIO.parse(handle, "fasta"):
        #print(f"ID: {record.id}")
        #print(f"Sequence: {record.seq}")
        #print(f"Description: {record.description}")
        #print("-" * 20)  # Separator for clarity

In [25]:
# Replace 'input.fasta' with the path to your input FASTA file
input_fasta_file = "/home/wenyuantong/Desktop/database/mobidb_search_2025-10-07T22-18-23.fasta"
# Replace 'filtered_output.fasta' with the desired name for the output file
output_fasta_file = "/home/wenyuantong/Desktop/database/mobidb_filtered_arabi.fasta"

sequences_to_keep = []

try:
    with open(input_fasta_file, "r") as infile:
        for record in SeqIO.parse(infile, "fasta"):
            # Check if the sequence contains '0' or '1'
            if '|sequence' in record.id:
                if len(record.id) < 50:
                    sequences_to_keep.append(record)
                else:
                    record.id = record.id[:50]
                    sequences_to_keep.append(record)
                    

    if sequences_to_keep:
        with open(output_fasta_file, "w") as outfile:
            SeqIO.write(sequences_to_keep, outfile, "fasta")
        print(f"Filtered sequences saved to '{output_fasta_file}'")
    else:
        print("No sequences without '0' or '1' were found to save.")

except FileNotFoundError:
    print(f"Error: Input file '{input_fasta_file}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Filtered sequences saved to '/home/wenyuantong/Desktop/database/mobidb_filtered_arabi.fasta'


In [26]:
    import subprocess
    import os

    # Define paths and filenames
    fasta_file = "/home/wenyuantong/Desktop/database/mobidb_filtered_arabi.fasta"
    db_name = "MobiDB_arabidopsis"
    db_type = "prot"  # or "prot" for protein sequences

    # Create a dummy FASTA file for demonstration
    #with open(fasta_file, "w") as f:
        #f.write(">seq1\nAGCTAGCTAGCT\n>seq2\nGCATGCATGCAT\n")

    # Construct the makeblastdb command
    # -in: input FASTA file
    # -dbtype: type of sequences (nucl for nucleotide, prot for protein)
    # -out: name of the database
    # -parse_seqids: enables parsing of sequence IDs (optional, but recommended)
    command = [
        "/home/wenyuantong/Desktop/packages/ncbi-blast-2.17.0+-x64-linux/ncbi-blast-2.17.0+/bin/makeblastdb",
        "-in", fasta_file,
        "-dbtype", db_type,
        "-out", db_name,
        "-parse_seqids"
    ]

    # Execute the command
    try:
        result = subprocess.run(command, check=True, capture_output=True, text=True)
        print("BLAST database created successfully.")
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Error creating BLAST database: {e}")
        print("STDOUT:", e.stdout)
        print("STDERR:", e.stderr)
    except FileNotFoundError:
        print("Error: makeblastdb not found. Ensure NCBI BLAST+ is installed and in your PATH.")

    # Clean up the dummy FASTA file
    #os.remove(fasta_file)

BLAST database created successfully.
STDOUT: 

Building a new DB, current time: 10/07/2025 20:28:57
New DB name:   /home/wenyuantong/jupyter_files/MobiDB_arabidopsis
New DB title:  /home/wenyuantong/Desktop/database/mobidb_filtered_arabi.fasta
Sequence type: Protein
Deleted existing Protein BLAST database named /home/wenyuantong/jupyter_files/MobiDB_arabidopsis
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 39276 sequences in 0.803727 seconds.



STDERR: 


In [28]:
import subprocess
import sys

# Define the paths to your files and database
# Make sure these paths are correct for your system
query_fasta_file = "/home/wenyuantong/Desktop/data/filtered_IDR_Arabi.fasta" # Replace with your query FASTA file path
database_name = '/home/wenyuantong/jupyter_files/MobiDB_arabidopsis'               # Replace with the base name of your custom database
output_file = 'blast_results.txt'            # Replace with the desired output file name

# Construct the blastp command
blastp_command = [
    '/home/wenyuantong/Desktop/packages/ncbi-blast-2.17.0+-x64-linux/ncbi-blast-2.17.0+/bin/blastp',
    '-query', query_fasta_file,
    '-db', database_name,
    '-out', output_file,
    '-outfmt', '6' # Tabular output format
]

print(f"Running command: {' '.join(blastp_command)}")

try:
    # Run the command
    # capture_output=True captures stdout and stderr
    # text=True decodes stdout and stderr as text
    result = subprocess.run(blastp_command, capture_output=True, text=True, check=True)

    print("\nBLASTP command executed successfully.")
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)

except FileNotFoundError:
    print(f"Error: blastp command not found. Make sure BLAST+ is installed and in your system's PATH.", file=sys.stderr)
except subprocess.CalledProcessError as e:
    print(f"Error executing blastp command: {e}", file=sys.stderr)
    print("STDOUT:", e.stdout, file=sys.stderr)
    print("STDERR:", e.stderr, file=sys.stderr)
except Exception as e:
    print(f"An unexpected error occurred: {e}", file=sys.stderr)

Running command: /home/wenyuantong/Desktop/packages/ncbi-blast-2.17.0+-x64-linux/ncbi-blast-2.17.0+/bin/blastp -query /home/wenyuantong/Desktop/data/filtered_IDR_Arabi.fasta -db /home/wenyuantong/jupyter_files/MobiDB_arabidopsis -out blast_results.txt -outfmt 6

BLASTP command executed successfully.
STDOUT:

STDERR:



In [31]:
import pandas as pd

# Replace 'blast_results.txt' with the actual path to your BLAST results file
blast_results_file = 'blast_results.txt'

# Define the column names for the tabular output (-outfmt 6)
# These are the standard column names. Adjust if you used a different outfmt.
column_names = [
    'qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen',
    'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore'
]


In [34]:
try:
    # Read the tabular BLAST results into a pandas DataFrame
    # The separator is typically a tab ('\t') for outfmt 6
    blast_df = pd.read_csv(blast_results_file, sep='\t', names=column_names)

    print("BLAST results loaded into a DataFrame:")
    display(blast_df.head())

    # You can now explore the DataFrame
    print("\nDataFrame Info:")
    blast_df.info()

    print("\nDescriptive Statistics for numerical columns:")
    display(blast_df.describe())

    # Example: Filter for hits with an E-value less than a threshold
    evalue_threshold = 0.05 # Example threshold
    significant_hits = blast_df[blast_df['evalue'] < evalue_threshold]
    print(f"\nSignificant hits (E-value < {evalue_threshold}):")
    display(significant_hits.head())

    # Example: Sort results by E-value (lowest E-value first)
    sorted_results = blast_df.sort_values(by='evalue')
    print("\nResults sorted by E-value:")
    display(sorted_results.head())


except FileNotFoundError:
    print(f"Error: BLAST results file '{blast_results_file}' not found. Please replace with the correct file path.")
except Exception as e:
    print(f"An error occurred while loading or exploring the results: {e}")

BLAST results loaded into a DataFrame:


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,Pap1089_5_ShcF1_3,Q94BP0|sequence|Probable,42.857,35,19,1,1,35,147,180,6.40,23.1
1,PapICMP1657_ShcF1_3,Q94BP0|sequence|Probable,42.857,35,19,1,1,35,147,180,6.40,23.1
2,PapICMP1659_ShcF1_3,Q94BP0|sequence|Probable,42.857,35,19,1,1,35,147,180,6.40,23.1
3,Pma88_10_ShcN1_1,Q9SV09|sequence|Zinc,31.250,48,33,0,22,69,223,270,0.62,28.1
4,Pma88_10_ShcN1_1,A0A1P8B846|sequence|Nudix,61.111,18,7,0,25,42,29,46,0.71,28.1



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166078 entries, 0 to 166077
Data columns (total 12 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   qseqid    166078 non-null  object 
 1   sseqid    166078 non-null  object 
 2   pident    166078 non-null  float64
 3   length    166078 non-null  int64  
 4   mismatch  166078 non-null  int64  
 5   gapopen   166078 non-null  int64  
 6   qstart    166078 non-null  int64  
 7   qend      166078 non-null  int64  
 8   sstart    166078 non-null  int64  
 9   send      166078 non-null  int64  
 10  evalue    166078 non-null  float64
 11  bitscore  166078 non-null  float64
dtypes: float64(3), int64(7), object(2)
memory usage: 15.2+ MB

Descriptive Statistics for numerical columns:


,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
count,166078.000000,166078.000000,166078.000000,166078.000000,166078.000000,166078.000000,166078.000000,166078.000000,1.660780e+05,166078.000000
mean,38.112060,40.682468,24.437475,0.674436,29.365894,67.884765,306.737997,345.365244,4.730363e+00,26.124765
std,9.963515,21.660727,14.721043,0.972177,36.799011,46.415170,379.315362,381.081442,2.845631e+00,2.556928
min,15.854000,7.000000,0.000000,0.000000,1.000000,10.000000,1.000000,10.000000,5.170000e-14,21.900000
25%,30.882000,26.000000,14.000000,0.000000,6.000000,36.000000,75.000000,112.000000,2.300000e+00,24.300000
50%,36.667000,35.000000,21.000000,0.000000,16.000000,52.000000,192.000000,232.000000,4.500000e+00,25.800000
75%,42.857000,50.000000,32.000000,1.000000,35.000000,83.000000,394.000000,430.000000,7.100000e+00,27.300000
max,100.000000,247.000000,164.000000,9.000000,370.000000,419.000000,4972.000000,5113.000000,1.000000e+01,70.100000



Significant hits (E-value < 0.05):


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
133,Pac302273_HopAP1_1,Q9LD18|sequence|Potassium,33.333,54,31,2,6,59,671,719,0.007,33.9
138,PacA10853_HopAP1_1,Q9LD18|sequence|Potassium,35.185,54,30,2,6,59,671,719,0.002,35.4
141,PacICMP2802_HopAP1_1,Q9LD18|sequence|Potassium,35.185,54,30,2,6,59,671,719,0.002,35.4
149,Pav013_HopAP1_1,Q9LD18|sequence|Potassium,35.185,54,30,2,6,59,671,719,0.002,35.4
152,PcbICMP2821_HopAP1_1,Q9LD18|sequence|Potassium,33.333,54,31,2,6,59,671,719,0.005,34.3



Results sorted by E-value:


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
11199,PcdICMP12471_HopI1_1,Q9LH49|sequence,49.231,65,32,1,140,204,10,73,5.170000e-14,68.9
10849,PcdICMP12341_HopI1_1,Q9LH49|sequence,49.231,65,32,1,140,204,10,73,5.170000e-14,68.9
12926,Pci0788_9_HopI1_1,Q9LH49|sequence,49.231,65,32,1,140,204,10,73,5.170000e-14,68.9
10850,PcdICMP12341_HopI1_1,A0A1I9LLP0|sequence,49.231,65,32,1,140,204,46,109,7.900000e-14,68.9
12927,Pci0788_9_HopI1_1,A0A1I9LLP0|sequence,49.231,65,32,1,140,204,46,109,7.900000e-14,68.9


In [38]:
display(significant_hits)

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
133,Pac302273_HopAP1_1,Q9LD18|sequence|Potassium,33.333,54,31,2,6,59,671,719,0.007,33.9
138,PacA10853_HopAP1_1,Q9LD18|sequence|Potassium,35.185,54,30,2,6,59,671,719,0.002,35.4
141,PacICMP2802_HopAP1_1,Q9LD18|sequence|Potassium,35.185,54,30,2,6,59,671,719,0.002,35.4
149,Pav013_HopAP1_1,Q9LD18|sequence|Potassium,35.185,54,30,2,6,59,671,719,0.002,35.4
152,PcbICMP2821_HopAP1_1,Q9LD18|sequence|Potassium,33.333,54,31,2,6,59,671,719,0.005,34.3
...,...,...,...,...,...,...,...,...,...,...,...,...
162587,PttICMP459_HrpH1_1,F4J5S1|sequence|Clustered,25.362,138,85,4,2,125,306,439,0.020,35.0
162683,PumICMP3962_HopAK1_3,Q9FNX5|sequence|Phragmoplastin,56.522,23,10,0,63,85,518,540,0.044,33.1
162831,PumICMP5931_HopAK1_1,Q9FNX5|sequence|Phragmoplastin,56.522,23,10,0,63,85,518,540,0.044,33.1
163309,PvrICMP19473_HrpQ1_1,Q9SID3|sequence|Hydroxyacylglutathione,27.429,175,94,9,6,171,120,270,0.015,35.8


In [36]:
# Find the number of unique values in 'col1'
unique_count_col1 = significant_hits['qseqid'].nunique()
print(f"\nNumber of unique values in 'col1': {unique_count_col1}")


Number of unique values in 'col1': 515


In [39]:
# Find the number of unique values in 'col1'
unique_count_col1 = significant_hits['sseqid'].nunique()
print(f"\nNumber of unique values in 'col1': {unique_count_col1}")


Number of unique values in 'col1': 240
